# 04 · Far-OOD: objects with no look-alike in PointLLM's training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Md-Mosabbir/localpoint3d/blob/main/notebooks/04_pointllm_far_ood.ipynb)

Notebook 03 showed that PointLLM files **near-OOD** objects under a familiar look-alike: rehal → bench, patil → bowl. This notebook adds **far-OOD** objects. They have no common look-alike in PointLLM's 658k training captions:

| Object | What it is | Closest training words |
|---|---|---|
| tubewell | hand-pump tube well | "tube well" 0, "hand pump" 4 |
| sheel pata | flat grinding stone + roller | "grinding stone" 0 |
| rickshaw | cycle rickshaw (borderline) | "rickshaw" 23, "tricycle" 71 |

We compare three groups:
- **everyday**: in-distribution, objects PointLLM knows well.
- **near-OOD**: a new name, but a look-alike exists.
- **far-OOD**: nothing similar exists.

If our grouping is right:
- far-OOD scans should be **farthest** from the everyday scans in PointLLM's fingerprint space,
- PointLLM should be **least sure** about them,
- and its answers should be the most scattered.

**What happens in this notebook**
0. **Settings** (the next cell)
1. Set up PointLLM (the same as notebook 03)
2. Load the scans of all 3 groups and the answer options
3. **OOD check:**
   - (a) training words
   - (b) fingerprint distance to the everyday scans
   - (c) a map of all fingerprints
4. **Multiple choice:** each scan's own 4 options, plus a ball and noise as blind inputs
5. **Confusion matrix:** every scan against **every** label, so we see what each object is mistaken for. How sure PointLLM is about its best label is a second OOD score.
6. **Report:** `report.md` + `report.pdf` + all tables and charts, saved together in one folder

**Before you start:** *Runtime → Change runtime type → T4 GPU*, check the Settings cell, then *Runtime → Run all*. Setup takes ~15–20 min, the tests ~20–30 min.

---
# ⚙️ Settings: change things here

**Which scans:** three folders, one per group.
- `EVERYDAY_DIR`: the 17 cleaned chairs/cups/tables/keyboards.
- `NEAR_DIR`: the cleaned local scans (bodna, piri, …).
- `FAR_DIR`: the far-OOD scans. Names like `Tubewell_1_cleaned.ply`.
- If the **same file name** is in two folders it is used once, and the far-OOD copy wins.
- `GROUP_OVERRIDES` puts an object in a group whatever folder it's in. From the training-word check (journal 05):
  - tubewell, sheel pata and rickshaw → **far_ood**
  - wooden bench → **everyday** (a bench is well known)
  - Remove a line to stop that override.
- `ONLY_SCANS`: `[]` = all. `["Tubewell", "Bodna"]` = only near/far files whose name contains these words.
- `COLOUR = "grey"`: shape only, as in notebook 03.

**Tests**
- `SEEDS`: tries per question for the multiple choice. PointLLM is random inside; we take the majority.
- `CONFUSION_SEEDS`: tries for the all-labels test. Each scan is scored against ~40 labels, so `[0]` keeps it at ~10 min.
- `BLIND_INPUTS`: a ball and random noise. They show what PointLLM says about "nothing", and they're needed for the calibrated score.

**Output:** everything goes into one folder, `RESULTS_DIR/<RUN_NAME>_<date>/`:
- `report.md`, `report.pdf`
- `figures/`
- `tables/`

> 🧠 A Python **dict** like `{"tubewell": "far_ood"}` maps a key to a value. Here: object name → group.

In [ ]:
# ---- Which scans
USE_DRIVE = True                                                # False = upload far-OOD scans by hand
EVERYDAY_DIR = "/content/drive/MyDrive/Models/cleared_test_data"                       # in-distribution reference (17 cleaned everyday scans)
NEAR_DIR = "/content/drive/MyDrive/Models/cleaned_bd/near_ood_set"                     # near-OOD scans (bodna, piri, ...)
FAR_DIR = "/content/drive/MyDrive/Models/cleaned_bd/far-OOD Ply"                       # far-OOD scans (tubewell, sheel pata, rickshaw)
GROUP_OVERRIDES = {"tubewell": "far_ood", "sheel_pata": "far_ood", "rickshaw": "far_ood",
                   "wooden_benched": "everyday"}               # object -> group, wins over the folder
RESULTS_DIR = "/content/drive/MyDrive/localpoint3d/results"    # a run folder is made inside
RUN_NAME = "far_ood"
ONLY_SCANS = []                                                 # [] = all near/far scans
SPECKS = "removed"
COLOUR = "grey"                                                 # "grey" (0.5), "black" (0) or "original"

# ---- Tests
OPTIONS_FILE = "data/mcq_options.csv"     # answer options per object (shared with notebook 03)
QUESTION = "What is this?"
SEEDS = [0, 1, 2]                         # multiple choice: tries per question (majority vote)
CONFUSION_SEEDS = [0]                     # all-labels test: tries per scan
BLIND_INPUTS = ["sphere", "noise"]        # what PointLLM says about "nothing"
SCORE_BY = "log_prob"                     # plain likelihood (see notebook 03)

# ---- OOD check
SEARCH_CAPTIONS = True                    # count words in PointLLM's training captions (227 MB download)
CAPTION_WORDS = ["tubewell", "tube well", "hand pump", "fire hydrant", "sheel", "grinding stone", "mortar",
                 "rickshaw", "tricycle", "cart", "bodna", "piri", "rehal", "bench", "pot", "chair"]
K_NEIGHBOURS = 3

---
# Part 1 · Setup

## Check the GPU

`nvidia-smi` lists the graphics card. We need roughly 14 GB of GPU memory for the 7B model in float16; a T4 has 15 GB. `free -h` shows normal RAM.

> 🧠 Lines starting with `!` are not Python. They are run as shell (terminal) commands.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!free -h | head -2

## Get our project code

Colab only opened this *notebook*. Our helper code lives in the repo's `src/` folder, so we clone the repo into the Colab machine (or update it if it's already there).

> 🧠 **Python note:** `sys.path` is the list of folders Python searches when you write `import something`. Adding `src/` to it is what lets us `import pc_io` below.
>
> 🧠 After a `!` command, `_exit_code` holds its exit code: 0 means success, anything else means it failed. We stop here if it failed, so the notebook never runs with an old copy of the code.

**The repo is private**, so Colab needs a GitHub **token** (a password just for this) to download it. Do this once per person:
1. GitHub → your picture → *Settings* → *Developer settings* → *Personal access tokens* → *Fine-grained tokens* → *Generate new token*.
2. *Repository access*: *Only select repositories* → `localpoint3d`. *Permissions* → *Contents*: **Read-only**. Generate, then copy the token.
3. In Colab: the 🔑 key icon on the left → *Add new secret* → Name `GITHUB_TOKEN`, Value = the token → switch on *Notebook access*.

Colab secrets belong to your Google account, so every notebook can use them, and they are never saved inside the notebook. Teammates need to be collaborators on the repo for their own token to work.


In [ ]:
import base64, os, sys

REPO_DIR = "/content/localpoint3d"

# The repo is private, so git needs a GitHub token: 🔑 Secrets panel -> GITHUB_TOKEN (see the text above).
# The token goes in a request header, not in the URL, so it never shows up in any output.
auth = ""
try:
    from google.colab import userdata
    basic = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()
    auth = f'-c http.extraHeader="Authorization: Basic {basic}"'
    print("Using the GITHUB_TOKEN secret")
except Exception as e:
    print("No GITHUB_TOKEN secret found:", type(e).__name__, "-> only works if the repo is public")

if os.path.isdir(REPO_DIR):
    !git {auth} -C {REPO_DIR} pull --ff-only
else:
    !git {auth} clone --depth 1 https://github.com/Md-Mosabbir/localpoint3d.git {REPO_DIR}
if _exit_code != 0:  # the exit code of the git command above: 0 = it worked
    raise RuntimeError("Downloading our code from GitHub failed (read the git message above). "
                       "Private repo: add the GITHUB_TOKEN secret (see the text above), then run this cell again.")

sys.path.insert(0, f"{REPO_DIR}/src")
print("Code is in", REPO_DIR)

## Get PointLLM

This is the official PointLLM code. We use HTTPS because the `git@…` SSH link in their README doesn't work on Colab.

In [ ]:
POINTLLM_DIR = "/content/PointLLM"
if not os.path.isdir(POINTLLM_DIR):
    !git clone --depth 1 https://github.com/InternRobotics/PointLLM.git {POINTLLM_DIR}
else:
    print("PointLLM already cloned")

## Build a separate Python 3.10 environment

PointLLM was written for **Python 3.10 + PyTorch 2.0.1 + an old Transformers**. Colab's own Python is newer and those versions won't install into it. So we make a separate, isolated Python 3.10 environment (a *virtual environment*, "venv") at `/content/venv` using `uv`, a fast installer.

The notebook itself keeps running on Colab's Python. Later we start the model **inside the venv** as a separate background process.

> 🧠 `%%bash` at the top turns the whole cell into a shell script. `set -e` = stop at the first error. `if [ -d … ]` checks whether the folder already exists, so running the cell twice in one session doesn't fail.

In [ ]:
%%bash
set -e
pip install uv
if [ -d /content/venv ]; then
    echo "/content/venv already exists (this session ran the cell before), keeping it"
else
    uv venv --python 3.10 /content/venv
fi


## Install PointLLM's dependencies into that environment (~5 min)

- `torch==2.0.1` matches what PointLLM was tested with
- `-e .` installs the PointLLM repo itself plus everything in its requirements
- `plyfile`, `trimesh` are for reading our scan files (used by `src/pc_io.py`)
- `numpy<2` everywhere because the old libraries break with NumPy 2

In [ ]:
%%bash
set -e
cd /content/PointLLM
PY=/content/venv/bin/python
uv pip install --python $PY "torch==2.0.1" "numpy<2"
uv pip install --python $PY -e . "numpy<2"
uv pip install --python $PY accelerate plyfile trimesh matplotlib "numpy<2"

## Smoke test

Quick check that everything imports **before** we spend 10 minutes downloading model weights. You should see `imports OK`. Warnings about deprecated functions are fine.

In [ ]:
%%bash
cd /content/PointLLM
/content/venv/bin/python -c "
import torch, transformers, pointllm, plyfile, trimesh
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)
print('imports OK')"

---
# Part 2 · Our scans

## Install the file readers in Colab's own Python

The venv has them already. The notebook's Python needs them too, so we can preview scans here. `reportlab` writes the PDF report at the end.

In [ ]:
!pip install plyfile trimesh reportlab

## Connect Google Drive

Our scans live on Google Drive. *Mounting* Drive makes it appear inside Colab as a normal folder, `/content/drive/MyDrive`. A Google popup asks for permission: allow it.

No Drive? Set `USE_DRIVE = False` in the Settings cell and a file picker opens to upload scans from your computer instead (they disappear when the session ends).

In [ ]:
if USE_DRIVE:  # set in the Settings cell
    from google.colab import drive
    drive.mount("/content/drive")
else:
    from google.colab import files
    os.makedirs("/content/scans", exist_ok=True)
    for name, data in files.upload().items():  # opens a file picker
        with open(os.path.join("/content/scans", name), "wb") as fh:
            fh.write(data)

## Find the scans and sort them into groups

1. Looks in the three folders: far first, then near, then everyday.
2. Skips a file name it has already seen, so a copy in two folders is used only once.
3. Applies `GROUP_OVERRIDES`.
4. Makes this run's output folder, `RUN_DIR`, with `figures/` and `tables/` inside.

In [ ]:
import glob
from datetime import datetime
import pandas as pd
from IPython.display import display
import charts, mcq_analysis, ood_analysis
from ood_analysis import GROUPS
from pc_io import parse_scan_name

if not USE_DRIVE:
    EVERYDAY_DIR, NEAR_DIR, FAR_DIR, RESULTS_DIR = "", "", "/content/scans", "/content/results"
RUN_STAMP = f"{RUN_NAME}_{datetime.now():%Y-%m-%d_%H%M}"
RUN_DIR = f"{RESULTS_DIR}/{RUN_STAMP}"
FIG_DIR, TABLE_DIR = f"{RUN_DIR}/figures", f"{RUN_DIR}/tables"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
EXTS = (".ply", ".obj", ".glb", ".gltf", ".off", ".stl")


def find_scans(folder):
    if not folder:
        return []
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"No folder at {folder}. Fix the path in the Settings cell (copy it from the Files panel).")
    return sorted(p for p in glob.glob(f"{folder}/**/*", recursive=True) if p.lower().endswith(EXTS))


def object_of(path):
    return parse_scan_name(path)["object"]


def save_figure(figure, name):
    """Save a chart or picture to FIG_DIR; returns the path relative to RUN_DIR (used by the report)."""
    if hasattr(figure, "savefig"):
        charts.save(figure, f"{FIG_DIR}/{name}.png")
    else:
        figure.save(f"{FIG_DIR}/{name}.png")
    return f"figures/{name}.png"


def save_table(df, name):
    df.to_csv(f"{TABLE_DIR}/{name}.csv", index=False)
    return f"tables/{name}.csv"


found = {}  # file name -> (path, group from its folder)
for group, folder in [("far_ood", FAR_DIR), ("near_ood", NEAR_DIR), ("everyday", EVERYDAY_DIR)]:
    for p in find_scans(folder):
        name = os.path.basename(p)
        if name in found:
            print(f"   {name} is also in the {group} folder: using it once, as {found[name][1]}")
            continue
        if ONLY_SCANS and group != "everyday" and not any(w in name for w in ONLY_SCANS):
            continue
        found[name] = (p, group)

GROUP_OF = {p: GROUP_OVERRIDES.get(object_of(p), g) for p, g in found.values()}
SCANS = {g: sorted(p for p, gg in GROUP_OF.items() if gg == g) for g in GROUPS}
ALL_SCANS = [p for g in GROUPS for p in SCANS[g]]
for g in GROUPS:
    print(f"{g}: {len(SCANS[g])} scan(s)")
    for p in SCANS[g]:
        print(f"   {os.path.basename(p):34s} -> {object_of(p)}")
if len(SCANS["everyday"]) < 2:
    print("⚠️ Need at least 2 everyday scans as the reference. Check EVERYDAY_DIR.")
print("\nThis run's folder:", RUN_DIR)

## Load the answer options

The same file as notebook 03, `data/mcq_options.csv`. It has rows for the far-OOD objects (tubewell → *hand pump / fire hydrant / faucet / chair*, …). Objects without a row are skipped in Parts 4–5, but they are still in the OOD check.

The all-labels test (Part 5) uses **every** answer text in the file that belongs to an object in this run, about 40 labels.

In [ ]:
options_path = OPTIONS_FILE if os.path.isabs(OPTIONS_FILE) else f"{REPO_DIR}/{OPTIONS_FILE}"
OPTIONS = pd.read_csv(options_path).set_index("object")
ROLES = [c for c in OPTIONS.columns if c != "note"]
CHANCE = 1 / len(ROLES)
PRESENT = [o for g in GROUPS for o in sorted({object_of(p) for p in SCANS[g]})]
missing = [o for o in PRESENT if o not in OPTIONS.index]
if missing:
    print("⚠️ No options for", missing, "-> skipped in Parts 4-5. Add rows to", OPTIONS_FILE)
OBJECTS = [o for o in PRESENT if o in OPTIONS.index]           # in group order
GROUP_OF_OBJECT = {object_of(p): g for p, g in GROUP_OF.items()}
CORRECT_LABEL = {o: OPTIONS.loc[o, "correct"] for o in OBJECTS}
LABELS = sorted({OPTIONS.loc[o, r] for o in OBJECTS for r in ROLES})
for obj, words in mcq_analysis.length_warnings(OPTIONS.loc[OBJECTS], ROLES):
    print(f"⚠️ Options for '{obj}' differ a lot in length {words}")


def as_answer(option):
    """'teapot' -> 'A teapot.'  (PointLLM's short training captions look like that)."""
    return f"{'An' if option[0].lower() in 'aeiou' else 'A'} {option}."


display(OPTIONS.loc[OBJECTS, ROLES])
print(f"{len(LABELS)} labels in the all-labels test:", ", ".join(LABELS))

## See what the model will see

One picture per near- and far-OOD object, in `COLOUR`, saved to `figures/` for the report. Check that each object is upright and clean (no floor or wall). An uncleaned scan would make the OOD check measure the clutter instead of the object.

In [ ]:
from pointllm_helpers import preprocess, preview

PICTURES = {}
for g in ["near_ood", "far_ood"]:
    for p in SCANS[g]:
        if object_of(p) not in PICTURES:
            picture = preview(preprocess(p, colour=COLOUR, specks=SPECKS)[0], title=f"{os.path.basename(p)} · {g}")
            display(picture)
            PICTURES[object_of(p)] = save_figure(picture, f"scan_{object_of(p)}")

---
# Part 3 · OOD check

## (a) Did PointLLM ever read these words?

Counts how many of PointLLM's 658,577 training captions contain each word in `CAPTION_WORDS`.
- Far-OOD objects should have no common look-alike word either.
- Near-OOD objects have a look-alike (bench, pot) with thousands of captions.

> 🧠 `\b` in a search pattern means "word boundary", so "cart" doesn't match "cartoon".

In [ ]:
import json, re
import matplotlib.pyplot as plt

word_counts = None
if SEARCH_CAPTIONS:
    from huggingface_hub import hf_hub_download
    caption_file = hf_hub_download("RunsenXu/PointLLM", "PointLLM_brief_description_660K_filtered.json",
                                   repo_type="dataset")
    with open(caption_file) as fh:
        training = json.load(fh)
    captions = [t["value"].lower() for item in training for t in item["conversations"] if t["from"] == "gpt"]
    word_counts = pd.DataFrame([{"word": w, "captions": sum(1 for c in captions if re.search(rf"\b{re.escape(w.lower())}\b", c))}
                                for w in CAPTION_WORDS])
    display(word_counts)
    save_table(word_counts, "training_words")
    FIG_WORDS = save_figure(charts.caption_counts(word_counts), "1_training_words")
    plt.show()

## Start the model in the background

Same as notebook 01: `src/pointllm_worker.py` runs in the Python 3.10 environment and keeps the model loaded. This notebook asks it for three things:
- `embed_pointllm(path)`: a **fingerprint** of the scan. PointLLM's point encoder turns the cloud into 513 tokens of 384 numbers, and we average them into one vector.
- `score_pointllm(path, answers)`: how likely PointLLM finds each answer after seeing the scan (a **log-probability**: 0 = certain, more negative = less likely).
- `ask_pointllm(path, prompt)`: a normal written answer (only for the optional CircularEval).

All three use `COLOUR` and `SPECKS` from Settings. A path like `"fake:sphere"` means a blind input instead of a file.

> 🧠 A **log-probability** is the logarithm of a probability. Probabilities of long answers get tiny (0.001 × 0.02 × …), and logs turn those multiplications into sums, which computers handle better. −1 ≈ 37 %, −3 ≈ 5 %, −7 ≈ 0.1 %.

In [ ]:
import subprocess
import numpy as np


def start_worker():
    env = {**os.environ, "POINTLLM_DIR": POINTLLM_DIR}
    w = subprocess.Popen(["/content/venv/bin/python", "-u", f"{REPO_DIR}/src/pointllm_worker.py"],
                         cwd=POINTLLM_DIR, env=env, text=True, bufsize=1,
                         stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT)
    print("Loading the model (first time: downloads ~14 GB, be patient)...")
    for line in w.stdout:
        if "@@READY@@" in line:
            print("✅ Model ready.")
            return w
        if line.strip():
            print("  ", line.rstrip()[:200])
    raise RuntimeError("The worker stopped while loading. Read the log above.")


def send(request):
    """Send one request to the worker and wait for its result."""
    request = {"colour": COLOUR, "specks": SPECKS, "rotation": "None", "floor": "kept", **request}
    worker.stdin.write(json.dumps(request) + "\n")
    worker.stdin.flush()
    for line in worker.stdout:
        if line.startswith("@@RESULT@@"):
            res = json.loads(line[len("@@RESULT@@"):])
            if "error" in res:
                raise RuntimeError(res["error"])
            return res
    raise RuntimeError("The worker died. Re-run this cell to restart it.")


def embed_pointllm(path, seed=0):
    return np.array(send({"task": "embed", "path": path, "prompt": "", "seed": seed})["vector"])


def score_pointllm(path, answers, seed=0):
    return send({"task": "score", "path": path, "prompt": QUESTION, "answers": answers, "seed": seed})["scores"]


def ask_pointllm(path, prompt, seed=0):
    return send({"task": "ask", "path": path, "prompt": prompt, "seed": seed})["answer"]


if "worker" not in globals() or worker.poll() is not None:  # don't load twice
    worker = start_worker()

## (b) Fingerprint distance to the everyday scans

Each scan gets PointLLM's fingerprint: the average of its 513 point tokens, averaged over `SEEDS`. We measure the distance to its `K_NEIGHBOURS` nearest everyday scans. Everyday scans are compared with the *other* everyday scans.
- **OOD line:** farther than 95 % of the everyday scans.
- **Expected:** everyday < near-OOD < far-OOD.

The dot chart shows each group's spread and median, which is easier to read with only 3 far-OOD scans.

> 🧠 **Median** = the middle value when sorted. Unlike the average, one extreme scan can't drag it far.

In [ ]:
from tqdm.auto import tqdm

raw = {p: np.mean([embed_pointllm(p, s) for s in SEEDS], axis=0) for p in tqdm(ALL_SCANS, desc="fingerprints")}
FINGERPRINT = ood_analysis.normalise(raw, SCANS["everyday"])
ood, threshold = ood_analysis.knn_table(FINGERPRINT, GROUP_OF, object_of, k=K_NEIGHBOURS)
dist_by_group = ood_analysis.group_summary(ood, "knn_distance")
print(f"OOD line: distance > {threshold:.3f}")
display(dist_by_group.round(3))
display(ood.sort_values(["group", "knn_distance"]).round(3))
save_table(ood, "ood_distance")
save_table(dist_by_group.reset_index(), "ood_distance_by_group")

FIG_DIST = save_figure(charts.group_distances(ood, threshold), "2_ood_distance")
plt.show()
FIG_DIST_GROUPS = save_figure(charts.group_strip(
    ood, "knn_distance", "distance to nearest everyday scans", "Distance by group",
    "Each dot is one scan; the black line is the group median. Expected: everyday < near-OOD < far-OOD."),
    "3_ood_distance_by_group")
plt.show()

## (c) Map of all fingerprints

PCA squeezes the 384 numbers of each fingerprint into 2, so we can draw them.
- Close points = shapes PointLLM sees as similar.
- A far-OOD scan alone in a corner = a shape unlike anything else here.

In [ ]:
xy = ood_analysis.pca_2d([FINGERPRINT[p] for p in ALL_SCANS])
FIG_MAP = save_figure(charts.fingerprint_map(xy, [os.path.basename(p) for p in ALL_SCANS],
                                             [GROUP_OF[p] for p in ALL_SCANS]), "4_fingerprint_map")
plt.show()

---
# Part 4 · Multiple choice (each object's own 4 options)

The same test as notebook 03, now for all three groups. Each scan's options are its correct answer, a look-alike, a same-family object and an unrelated one, so guessing gets 25 %. The ball and noise get the same options.

In [ ]:
import random

jobs = []  # (group, object, input kind, path)
for g in GROUPS:
    scans = [p for p in SCANS[g] if object_of(p) in OPTIONS.index]
    jobs += [(g, object_of(p), "real", p) for p in scans]
    for obj in sorted({object_of(p) for p in scans}):
        for kind in BLIND_INPUTS:
            if kind == "other":
                others = [p for p in ALL_SCANS if object_of(p) != obj]
                jobs.append((g, obj, "other", random.Random(obj).choice(others)))
            else:
                jobs.append((g, obj, kind, f"fake:{kind}"))
n_mcq = len(jobs) * len(SEEDS) * len(ROLES)
n_all = (len([p for p in ALL_SCANS if object_of(p) in OPTIONS.index]) + 2) * len(CONFUSION_SEEDS) * len(LABELS)
print(f"Multiple choice: {n_mcq} model passes; all-labels test: {n_all}. "
      f"Together roughly {(n_mcq + n_all) * 0.4 / 60:.0f}-{(n_mcq + n_all) * 0.8 / 60:.0f} min on a T4")

## Run the multiple choice → table

One row per input × seed × answer, with its log-probability (closer to 0 = more likely).

In [ ]:
rows = []
for g, obj, kind, path in tqdm(jobs, desc="multiple choice"):
    options = [OPTIONS.loc[obj, r] for r in ROLES]
    answers = [as_answer(o) for o in options]
    for seed in SEEDS:
        for role, option, answer, (lp, nt) in zip(ROLES, options, answers, score_pointllm(path, answers, seed)):
            rows.append({"set": g, "object": obj, "input": kind,
                         "file": path if kind in ("sphere", "noise") else os.path.basename(path),
                         "colour": COLOUR, "seed": seed, "role": role, "option": option, "answer": answer,
                         "log_prob": lp, "n_tokens": nt})
mcq = pd.DataFrame(rows)
save_table(mcq, "mcq_scores")
print(len(mcq), "rows saved")

## Multiple-choice results

- `pick` = the most likely answer (majority over seeds). `seeds_agreeing` = how many seeds chose it.
- **Accuracy per group** uses the binomial test against 25 % guessing. The ball and noise rows show what PointLLM picks with no object at all.
- **Picks per group:** which *kind* of answer wins. Near-OOD went mostly to look-alikes in notebook 03. What does far-OOD go to?

In [ ]:
best = mcq_analysis.picks(mcq, SCORE_BY)
vote = mcq_analysis.vote(best, OPTIONS)
mcq_table = mcq_analysis.per_scan(vote)
mcq_summary = mcq_analysis.accuracy_summary(mcq_table, vote, CHANCE, BLIND_INPUTS)
display(mcq_table[["set", "file", "pick_text", "pick", "seeds_agreeing", "correct"]])
display(mcq_summary.round(3))
save_table(mcq_table, "mcq_per_scan")
save_table(mcq_summary, "mcq_summary")

FIG_MCQ = save_figure(charts.accuracy_by_input(mcq_summary, CHANCE), "5_mcq_accuracy")
plt.show()
FIG_PICKS = {}
for g in GROUPS:
    seed_picks = best[(best.set == g) & (best.input == "real")]
    if len(seed_picks):
        FIG_PICKS[g] = save_figure(charts.picks_by_object(seed_picks, ROLES, title=f"What does PointLLM pick? ({g})"),
                                   f"6_mcq_picks_{g}")
        plt.show()

---
# Part 5 · All-labels test and confusion matrix

In the multiple choice every object only saw its own 4 answers. Here **every scan is scored against every label** (~40: hand pump, fire hydrant, bench, bowl, chair, …). The best label is PointLLM's prediction. That gives a real **confusion matrix**:
- **Rows** = the true object, split into everyday / near / far.
- **Columns** = the label PointLLM picked.
- **Cells** = number of scans.
- The **outlined cell** is the correct label. A diagonal of outlined, filled cells = everything recognised.

We make it twice:
- **Plain:** the most likely label. Common, short words like "chair" have an advantage.
- **Calibrated:** each label's likelihood minus its likelihood for the ball/noise. That removes the advantage, so this one shows what the **shape** points to.

> 🧠 A **confusion matrix** counts, for every true class, how often each prediction was made. Off-diagonal cells are the confusions.

In [ ]:
label_answers = [as_answer(l) for l in LABELS]
inputs = [(GROUP_OF[p], object_of(p), "real", p) for p in ALL_SCANS if object_of(p) in OPTIONS.index]
inputs += [("blind", "-", k, f"fake:{k}") for k in ["sphere", "noise"]]
rows = []
for g, obj, kind, path in tqdm(inputs, desc="all labels"):
    for seed in CONFUSION_SEEDS:
        for label, (lp, nt) in zip(LABELS, score_pointllm(path, label_answers, seed)):
            rows.append({"group": g, "object": obj, "input": kind,
                         "file": path if kind != "real" else os.path.basename(path),
                         "seed": seed, "label": label, "log_prob": lp, "n_tokens": nt})
all_labels = ood_analysis.add_calibrated(pd.DataFrame(rows))
save_table(all_labels, "all_labels_scores")
print(len(all_labels), "rows saved")

## Confusion matrices

The **accuracy table** counts, per group, how often the correct label is the best one (top-1) and among the best three (top-3). Guessing would get top-1 right about 1 time in 40.

In [ ]:
PRED, CM, FIG_CM = {}, {}, {}
for score in ["log_prob", "calibrated"]:
    pred = ood_analysis.mark_correct(ood_analysis.predictions(all_labels, score), CORRECT_LABEL)
    PRED[score] = pred
    CM[score] = ood_analysis.confusion(pred, OBJECTS)
    name = "plain" if score == "log_prob" else "calibrated"
    save_table(pred.assign(top3=pred.top3.map(", ".join)), f"all_labels_predictions_{name}")
    save_table(CM[score].reset_index(), f"confusion_matrix_{name}")
    FIG_CM[score] = save_figure(charts.confusion_heatmap(
        CM[score], GROUP_OF_OBJECT, CORRECT_LABEL,
        title=f"Confusion matrix ({name}): what PointLLM calls each object"), f"7_confusion_{name}")
    plt.show()

ALL_ACC = pd.concat([
    PRED[s].groupby("group").agg(scans=("file", "count"), top1=("correct", "sum"), top3=("in_top3", "sum"))
    .reindex([g for g in GROUPS if g in set(PRED[s].group)]).assign(score="plain" if s == "log_prob" else "calibrated")
    for s in PRED]).reset_index()
display(ALL_ACC)
save_table(ALL_ACC, "all_labels_accuracy")
display(PRED["calibrated"][["group", "file", "predicted", "top3", "correct_label"]])

## How sure is PointLLM? A second OOD score

For each scan, `softmax` turns the likelihoods of all ~40 labels into shares that add up to 1. `max_share` is the share of the best label:
- **High:** one label clearly wins. The object feels familiar.
- **Low:** the votes are spread out. Unfamiliar.

This is the idea of the MCM out-of-distribution score (Ming et al. 2022). **Expected:** everyday highest, far-OOD lowest.

In [ ]:
conf = ood_analysis.confidence(all_labels)
conf_by_group = ood_analysis.group_summary(conf, "max_share")
display(conf_by_group.round(3))
save_table(conf, "confidence")
save_table(conf_by_group.reset_index(), "confidence_by_group")
FIG_CONF = save_figure(charts.group_strip(
    conf, "max_share", "share of the best label (of ~40)", "How sure is PointLLM about its best label?",
    "Each dot is one scan. Expected: everyday highest, far-OOD lowest."), "8_confidence_by_group")
plt.show()

---
# Part 6 · The report: Markdown + PDF + everything in one folder

Builds `report.md` and `report.pdf` in `RUN_DIR` from the results above:
- summary numbers
- the scans per group
- all charts
- the confusion matrices
- the key tables

`src/report_writer.py` writes both files from the same content, so they always match.

To put a run in the repo, copy the whole run folder from Drive into `results/far_ood/` and send it to Claude. The Markdown links still work after the copy because they are relative.

In [ ]:
from report_writer import Report


def med(table, value, group):
    t = table[table.group == group][value]
    return f"{t.median():.2f}" if len(t) else "-"


def acc_line(group):
    r = mcq_summary[(mcq_summary.set == group) & (mcq_summary.label == "real scans")]
    return f"{int(r.correct.iloc[0])}/{int(r.n.iloc[0])}" if len(r) else "-"


rep = Report("LocalPoint3D · Far-OOD test",
             f"Run {RUN_STAMP} · PointLLM 7B v1.2 · {COLOUR} · seeds {SEEDS} · {len(ALL_SCANS)} scans")
rep.heading("1. Summary")
order_ok = (ood.groupby("group").knn_distance.median().reindex(GROUPS).dropna().is_monotonic_increasing)
far_named = PRED["calibrated"][PRED["calibrated"].group == "far_ood"]
rep.bullets([
    "Scans: " + ", ".join(f"**{g}** {len(SCANS[g])}" for g in GROUPS) + ".",
    "Median fingerprint distance to the everyday scans: " + ", ".join(f"{g} {med(ood, 'knn_distance', g)}" for g in GROUPS)
    + (". **Ordered as expected** (everyday < near < far)." if order_ok else ". **Not** in the expected order."),
    "Multiple choice (own 4 options, 25 % by guessing), real scans correct: "
    + ", ".join(f"{g} {acc_line(g)}" for g in GROUPS) + ".",
    "All-labels test (~1 in " + str(len(LABELS)) + " by guessing), calibrated top-1 / top-3: "
    + ", ".join(f"{r.group} {int(r.top1)}/{int(r.scans)} / {int(r.top3)}/{int(r.scans)}"
                for r in ALL_ACC[ALL_ACC.score == "calibrated"].itertuples()) + ".",
    "How sure PointLLM is about its best label (median share): "
    + ", ".join(f"{g} {med(conf, 'max_share', g)}" for g in GROUPS) + ".",
    "Far-OOD objects were called: " + "; ".join(f"{r.object} -> {r.predicted}" for r in far_named.itertuples()) + ".",
])
rep.heading("2. What we ran")
rep.table(pd.DataFrame([{"group": g, "scans": len(SCANS[g]),
                          "objects": ", ".join(sorted({object_of(p) for p in SCANS[g]}))} for g in GROUPS]))
rep.text(f"Model RunsenXu/PointLLM_7B_v1.2 (T4, float16). Input: 8192 points, {COLOUR}, specks removed. "
         f"Question: \"{QUESTION}\". Answers are scored by likelihood, not generated. Multiple-choice seeds {SEEDS}; "
         f"all-labels seeds {CONFUSION_SEEDS}; blind inputs {BLIND_INPUTS}. Options: {OPTIONS_FILE}.")
for obj, path in PICTURES.items():
    rep.figure(path, f"{obj} ({GROUP_OF_OBJECT[obj]}): what the model sees")
rep.heading("3. Are the groups really different? (OOD check)")
if word_counts is not None:
    rep.figure(FIG_WORDS, "Training captions containing each word (of 658,577).")
rep.figure(FIG_DIST, f"Distance to the 3 nearest everyday scans. OOD line = {threshold:.3f}.")
rep.figure(FIG_DIST_GROUPS, "The same distances by group; black line = median.")
rep.table(dist_by_group)
rep.figure(FIG_MAP, "Map of PointLLM's fingerprints (PCA).")
rep.heading("4. Multiple choice (each object's own 4 options)")
rep.table(mcq_summary[["set", "label", "correct", "n", "accuracy", "p_value"]])
rep.figure(FIG_MCQ, "Real scans vs ball and noise. Dashed line = guessing (25 %).")
for g, path in FIG_PICKS.items():
    rep.figure(path, f"Which kind of answer wins ({g}).")
rep.table(mcq_table[["set", "file", "pick_text", "pick", "seeds_agreeing"]])
rep.heading("5. All-labels test and confusion matrix")
rep.table(ALL_ACC)
rep.figure(FIG_CM["log_prob"], "Plain likelihood. Outline = correct label.")
rep.figure(FIG_CM["calibrated"], "Calibrated (minus ball/noise). Outline = correct label.")
rep.table(PRED["calibrated"].assign(top3=PRED["calibrated"].top3.map(", "
          .join))[["group", "file", "predicted", "top3", "correct_label"]])
rep.heading("6. How sure is PointLLM? (confidence)")
rep.figure(FIG_CONF, "Share of the best label among all labels; lower = less sure.")
rep.table(conf_by_group)
rep.heading("7. How to read this")
rep.bullets([
    "**Far-OOD is confirmed** if far-OOD scans are farthest from the everyday scans, PointLLM is least sure about them, "
    "and its labels for them are scattered or unrelated.",
    "**Near-OOD** objects should go to their look-alike (rehal -> bench), with the shape still pointing that way after calibration.",
    "Small groups (3 far-OOD scans): read the dots, not just the medians. One scan can move a median.",
    "Raw tables are in tables/; every chart is in figures/.",
])
md_path, pdf_path = rep.save(RUN_DIR)
print("✅ Report written:\n  ", md_path, "\n  ", pdf_path)
print("\nEverything from this run:")
for root, _, files in os.walk(RUN_DIR):
    for f in sorted(files):
        print("  ", os.path.relpath(os.path.join(root, f), RUN_DIR))

---
## Stop the model (frees the GPU)

In [ ]:
worker.terminate()
print("Worker stopped.")

---
## How to read the results

| What you see | What it means |
|---|---|
| far-OOD dots highest in the distance chart, lowest in the confidence chart | Our far-OOD grouping is right: PointLLM finds these shapes unfamiliar |
| far-OOD rows of the confusion matrix spread over unrelated labels | It has nothing to map them to |
| near-OOD rows mostly in one look-alike column (bench, bowl) | The near-OOD pattern from notebook 03 |
| everyday rows on the outlined diagonal | The test works on known objects |
| far-OOD not farther than near-OOD | Maybe the "far" objects resemble something after all (e.g. sheel pata ≈ a slab), or a scan is not clean |

## Troubleshooting

| Problem | Try |
|---|---|
| `No folder at …` | Fix the three folder paths in Settings (copy them from the Files panel). |
| An object is missing from Parts 4–5 | It has no row in `data/mcq_options.csv`. The name is the part of the file name before the number. |
| A scan is in the wrong group | Add it to `GROUP_OVERRIDES`, e.g. `"thela_gari": "far_ood"`. |
| `PDF skipped` | Run `!pip install reportlab`, then re-run the report cell. |
| `CUDA out of memory` | Runtime → Restart session and run again. |